In [ ]:
import os
import warnings
import random
import pickle
from datetime import datetime
from collections import defaultdict
from functools import partial

# Environment configuration
os.environ['JAX_PLATFORMS'] = 'cpu'
os.environ['XLA_FLAGS'] = '--xla_force_host_platform_device_count=8'

import numpyro
from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS, log_likelihood, hmc
from numpyro.infer.util import initialize_model
from numpyro.util import fori_collect

import jax
from jax import config
config.update("jax_enable_x64", True)
config.update('jax_platform_name', 'cpu')

import jax.numpy as jnp
from jax import jit, pmap, devices, device_get, lax, local_device_count, random, vmap, block_until_ready
from jax.random import PRNGKey, split
from jax.scipy.optimize import minimize
from jax.scipy.signal import fftconvolve
from jax.scipy.signal import convolve as jax_convolve

# Check devices
print('Available devices:', devices())
print('CPU devices:', devices('cpu'))

# Suppress warnings
warnings.filterwarnings('ignore', message="It appears that you're using a Mac with one of Apple's ARM-based processors")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, to_hex
import corner

import astropy.units as u
import astropy.constants as c
from astropy.constants import G, m_p
from astropy.table import Table

import shone
# from shone.opacity.dace import download_molecule
# from shone.chemistry import FastchemWrapper
# from shone.opacity import Opacity
# from shone.transmission import de_wit_seager_2013

import fleck
from fleck.jax import ActiveStar

# from specutils.manipulation import box_smooth, gaussian_smooth, trapezoid_smooth
# from specutils.spectra import Spectrum1D, SpectralRegion

# from scipy.io import *
from scipy.optimize import fmin_powell, curve_fit
# from scipy.interpolate import interp1d
# from scipy.signal import correlate
from scipy.stats import gaussian_kde

# from astropy.convolution import convolve, convolve_fft
# from astropy.convolution import Gaussian1DKernel

# from IPython.display import display
# from ipywidgets import interactive, VBox, HBox, FloatSlider

from chromatic import *
from svo_filters import svo
from sphinx import get_interp_stellar_spectrum

import arviz
import arviz as az

from tqdm.auto import tqdm

example_sphinx_file = pd.read_csv('/Users/wiwa8630/model-spectra/sphinx/SPECTRA/Teff_2000.0_logg_4.0_logZ_-0.5_CtoO_0.3_spectra.txt',
                             comment='#',delimiter=r'\s+',
                             names=['wavelength', 'flux']
                             )
panchromatic_wavelengths = example_sphinx_file['wavelength'].values
half_diffs = np.diff(panchromatic_wavelengths) / 2.0
bin_edges = np.zeros(len(panchromatic_wavelengths) + 1)
bin_edges[0] = panchromatic_wavelengths[0] - half_diffs[0]
bin_edges[1:-1] = panchromatic_wavelengths[:-1] + half_diffs
bin_edges[-1] = panchromatic_wavelengths[-1] + half_diffs[-1]
panchromatic_bin_edges = bin_edges
panchromatic_sphinx_grid = get_interp_stellar_spectrum(panchromatic_bin_edges)

species = ['H2O', 'CH4', 'CO2', 'CO','NH3']

visits = {
    'F21': {
        'Grism': 'G141',
        # 'Forward': G_141_for_dict,
        # 'Backward': G_141_back_dict,
        'BJD_times': np.array(pd.read_csv('../../data/F21_bjdtimes.csv')['BJD'][:]) * u.day,
        'time_lower': 2459455.708 * u.day,
        'time_upper': 2459455.738 * u.day,
        'T0 (BJD_TDB)': 2459455.9895 * u.day,
        'exp (s)': 4.9784 * u.s,
        'native resolution': 46.3 * u.angstrom
    },
    'S22': {
        'Grism': 'G102',
        # 'Forward': G_102_for_dict,
        # 'Backward': G_102_back_dict,
        'BJD_times': np.array(pd.read_csv('../../data/S22_bjdtimes.csv')['BJD'][:]) * u.day,
        'time_lower': 2459684.215 * u.day,
        'time_upper': 2459684.243 * u.day,
        'T0 (BJD_TDB)': 2459684.4959 * u.day, # This is 27 planetary orbits after the first transit, + 0.0054 days (the transit arrived 7 minutes late)
        'exp (s)': 9.67632 * u.s,
        'native resolution': 24.6 * u.angstrom
    }
}

systeminfo = {
    'duration (hr)': 3.5 * u.hr,
    'T_orb (d)': 8.463 * u.day,
    'T_rot (d)': 4.86 * u.day,
    'inclination': 89.5,
    'eccentricity': 0.0,
    'longitude_of_periastron': 88.4
}

def read_sensitivity_curve(grism='G141'):
    path = f'../../data/WFC3.IR.{grism}.1st.sens.2.fits'

    response = fits.open(path)

    w = response[1].data['wavelength']/1e4 * u.micron
    s = response[1].data['sensitivity'] * u.cm * u.cm / u.erg
    e = response[1].data['error'] * u.cm * u.cm / u.erg
    
    return w, s, e

# https://shone.readthedocs.io/en/latest/shone/examples/transmission.html#general-transmission-spectra
def transmission_spectrum_HengKitz17(log_atm_pressure = -1,
                                     atm_temp = 600 * u.K,
                                     log_kappa_cloud = -2,  # cloud opacity [cm2 / g]
                                     mmw=2.5,
                                     R_0=3.5 * u.R_earth,
                                     M_p = 10.2 * u.M_earth, **kwargs):
    """
    Compute a transmission spectrum for an atmosphere
    using the isothermal/isobaric approximation
    from Heng & Kitzmann (2017).
    """

    P_0 = 10**log_atm_pressure
    temperature = jnp.array([atm_temp.value])  # [K]
    pressure = jnp.array([P_0]) # [bar]
    chem = FastchemWrapper(temperature, pressure)
    vmr = chem.vmr()
    weights = chem.get_weights()

    weighted_opacities = []
    for i, spec in enumerate(species):
        op = binned_opacities[i](atm_temp.value, P_0)[0]  # cm2 / g
        col_idx = chem.get_column_index(species_name=spec)[0]
        species_weight = weights[col_idx] / mmw
    
        abund_weighted_opacity = op * species_weight * vmr[:, col_idx]
        weighted_opacities.append(abund_weighted_opacity)

    total_mol_opacity = jnp.array(weighted_opacities).sum(axis=0)
    
    g = ( (c.G * M_p) / (R_0)**2 ).decompose() # surface gravity
    
    # compute the planetary radius as a function of wavelength:
    Rp = heng_kitzmann_2017.transmission_radius_isothermal_isobaric(
        total_mol_opacity + (10**log_kappa_cloud),
        R_0.cgs.value, P_0, atm_temp, mmw, g.cgs.value
    )

    # convert to transit depth:
    transit_depth_ppm = 1e6 * (Rp / (0.8*u.R_sun).cgs.value) ** 2
        
    return (Rp / (0.8*u.R_sun).cgs.value), transit_depth_ppm

# Example Usage:
# Rp, transit_depth_ppm = transmission_spectrum_HengKitz17()
# plt.plot(wavelengths, Rp)

@jit
def breathing_model_jax(phase, b1, b2, b3, b4):

    phase = jnp.array(phase)
    _breathing = 1. + (b1 * phase) + (b2 * phase**2.) + (b3 * phase**3.) + (b4 * phase**4.)
    breathing = _breathing/jnp.mean(_breathing)
    
    return jnp.array(breathing)

@jit
def ramp_model_jax(phase, r1, r2, r3):

    phase = jnp.array(phase)
    _ramp = 1. - jnp.exp( (-r1 * phase) + r2) + (r3 * phase)
    ramp = _ramp/jnp.mean(_ramp)
    
    return jnp.array(ramp)

@jit
def linear_model_jax(x, m):

    x = jnp.array(x)
    _line = m * x + 1
    line = _line/jnp.mean(_line)

    return jnp.array(line)

@jit
def get_planck_spectrum_jax(T, **kwargs):
    """
    Calculate the surface flux from a thermally emitted surface,
    according to Planck function.

    Parameters
    ----------
    wavelength : Quantity
        The wavelengths at which to calculate,
        with units of wavelength.
    temperature : Quantity
        The temperature of the thermal emitter,
        with units of K.

    Returns
    -------
    surface_flux : Quantity
        The surface flux, evaluated at the wavelengths.
    """

    # define variables as shortcut to the constants we need
    h = 6.62607e-27 # erg s
    k = 1.380649e-16 # erg/K
    c = 2.9979e18 # angstrom/s
    wavelength = panchromatic_wavelengths*1e4

    z = h * c / (wavelength * k * T) # units check out

    # calculate the intensity from the Planck function
    intensity = (2 * h * c**2 / wavelength**5 / (jnp.exp(z) - 1)) # Units are erg/s/A^3

    # calculate the flux assuming isotropic emission
    flux = jnp.pi * intensity * 1e16 # erg / (s * cm^2 * angstrom)

    # return the intensity
    wave_jax = jnp.array(panchromatic_wavelengths)
    flux_jax = jnp.array(flux)

    return wave_jax, flux_jax

@jit
def convolve_spectrum_jax(model_wavelength, model_flux, sigma, kernel_size=3, **kwargs):
    """
    Properly convolve a spectrum with a Gaussian kernel in JAX.
    
    Args:
        model_wavelength: Array of wavelengths (must be evenly spaced!)
        model_flux: Corresponding flux values
        sigma: Standard deviation of Gaussian kernel in wavelength units
        kernel_size: Number of elements in the kernel (odd number recommended)
        
    Returns:
        Convolved flux array
    """
    # Ensure inputs are JAX arrays
    model_wavelength = jnp.asarray(model_wavelength)
    model_flux = jnp.asarray(model_flux)
    
    # Create proper Gaussian kernel
    x = jnp.linspace(-(kernel_size//2), kernel_size//2, kernel_size)
    kernel = jnp.exp(-0.5 * (x/sigma)**2)
    kernel = kernel / jnp.sum(kernel)  # normalize
    
    # Perform convolution
    convolved = jax_convolve(model_flux, kernel, mode='same', method='fft')
    
    return convolved

@jit
def get_sphinx_spectrum_jax(T, metallicity, CtoO, grid=panchromatic_sphinx_grid, **kwargs):
    
    # Fix: Create a tuple of the three values, then convert to array
    gridspec = grid(
        jnp.array(T, dtype=jnp.float64),
        jnp.array(metallicity, dtype=jnp.float64),
        jnp.array(CtoO, dtype=jnp.float64)
    )
    
    # Calculate the normalization factor
    # sigma_sb = 5.67e-5  # erg/cm^2/s
    # nf = (sigma_sb * (T)**4) / (jnp.trapezoid(gridspec, x=jnp.array(panchromatic_wavelengths)*1e4))
    re_normed_flux = gridspec #* nf
    
    wave_jax = jnp.array(panchromatic_wavelengths)
    flux_jax = jnp.array(re_normed_flux)
    
    return wave_jax, flux_jax


F21_speclc_bin_edges = np.array([1.14296 ,
1.15263 ,
1.16318 ,
1.18179 ,
1.21123 ,
1.22782 ,
1.2374 ,
1.25408 ,
1.26804 ,
1.28748 ,
1.32515 ,
1.33291 ,
1.34682 ,
1.37287 ,
1.39266 ,
1.40199 ,
1.4164 ,
1.43361 ,
1.45982 ,
1.48938 ,
1.51175 ,
1.53161 ,
1.55509 ,
1.56757 ,
1.6138 ,
1.63867 ,]) * u.micron

F21_speclc_err_factor = np.array([1.00	,
1.07	,
1.41	,
2.17	,
1.30	,
1.00	,
1.28	,
1.18	,
1.66	,
1.86	,
1.00	,
1.21	,
1.98	,
1.51	,
1.00	,
1.27	,
1.43	,
1.89	,
2.01	,
1.72	,
1.71	,
1.61	,
1.30	,
2.54	,
1.69	,])


F21_SED_err_factor = np.array([11.606,10.016,3.183,2.000,2.718,3.906,6.409,
                               2.0,3.460,3.134,2.821,2.0,2.0,5.550,
                               2.273,2.000,2.000,2.0,5.025,4.385,2.0,
                               2.683,3.160,2.0,3.135,2.128,2.000,2.000,
                               2.0,2.694,3.583,2.0,2.0,3.989,3.019,
                               6.914,17.888,18.138,5.215,8.532,2.183,3.095,
                               2.0,2.0,4.225,5.362,2.000,2.000,2.0,
                               4.008,4.563,7.652,2.144,2.762,3.996,9.622,
                               2.0,2.0,4.172,2.0,2.942,3.088,3.457,
                               2.0,4.303,2.257,4.045,11.133,4.696,2.0,
                               2.0,2.184,2.651,2.481,2.0,3.845,2.0,
                               2.0,8.637,2.695,8.502,2.680,5.431,4.670,
                               2.0,2.0,2.0,2.0,3.382,6.319,2.400,
                               2.0,2.0,2.0,2.878,3.070,10.219,3.012,
                               2.082,7.564,11.554,5.829,2.976,3.876,4.837,
                               7.078,2.0,2.775,2.0,2.000,])


S22_speclc_bin_edges = np.array([0.8697709,
0.87960467,
0.88943845,
0.90418911,
0.91156444,
0.92139821,
0.93369043,
0.94844109,
0.96073331,
0.96810864,
0.97302553,
0.97794241,
0.98531775,
1.01481907,
1.02465284,
1.03202817,
1.03694506,
1.04432039,
1.05661261,
1.06644638,
1.07382171,
1.10086459,
1.11315681,
1.12299059,
1.13]) * u.micron

S22_speclc_err_factor = np.array([
1.00	,
1.00	,
1.00	,
1.00	,
1.00	,
1.00	,
1.00	,
1.55	,
1.00	,
1.00	,
1.00	,
1.00	,
1.00	,
1.00	,
1.00	,
1.00	,
1.00	,
1.00	,
1.00	,
1.00	,
2.93	,
1	,
1	,
1	,])

S22_SED_err_factor = ([1.100, 1.100,1.000,1.100,1.100,1.100,
                       1.100,1.100,1.100,1.100,1.187,1.341,1.100,
                       1.100,1.100,1.100,1.012,1.100,1.1,1.283,
                       1.125,1.272,1.362,1.922,1.976,2.007,1.206,
                       1.387,1.357,1.392,1.798,1.202,1.100,1.100,
                       1.1,1.100,1.223,1.100,1.120,1.131,1.100,
                       1.219,1.100,1.100,1.295,1.1,1.1,1.1,
                       1.100,1.100,1.100,1.000,1.1,1.487,1.265,
                       1.1,1.100,1.1,1.364,1.1,1.1,1.1,
                       1.100,1.100,1.069,1.237,1.486,1.222,1.1,
                       1.103,1.100,1.1,1.1,1.268,1.1,1.1,
                       1.1,1.100,1.1,1.223,1.781,1.470,1.1,
                       1.100,1.100,1.185,1.762,2.281,1.247,1.102,
                       1.1,1.482,1.221,1.1,1.118,1.100,1.108,
                       1.134,1.10,1.100,1.100,1.100,1.100,1.100,
                       1.224,1.1,1.1,1.100,1.1,1.1,1.10,
                       1.1,1.1,1.1,1.1,1.1,1.194,1.791,
                       2.815,1.795,1.1,1.321,1.827,1.840,1.222,
                       1.106,1.100,1.1,1.100,1.1,1.1,1.151,
                       1.221,1.100,1.135,1.115,1.100,1.100,])

In [ ]:
visit = 'S22'
err_factor = S22_SED_err_factor

'Things that automatically get re-defined for either visit'
predicted_T0 = visits[f'{visit}']['T0 (BJD_TDB)'].value
binwidth = visits[f'{visit}']['native resolution']
exptime = visits[f'{visit}']['exp (s)']
grism = visits[f'{visit}']['Grism']
rainbow = read_rainbow(f"../../data/rainbows/{visit}_scan-combined_trimmed_pacman_spec.rainbow.npy")
speclc_rainbow = rainbow

assert(len(err_factor)==len(speclc_rainbow.wavelength))

'Adjust uncertainties and bin the rainbow'
for i in range(len(rainbow.wavelength.value)):
    speclc_rainbow.uncertainty[i,:] = rainbow.uncertainty[i,:] #* err_factor[i]

'remove the transit'
out_of_transit = ( speclc_rainbow.trim().mask_transit(period=8.463*u.day, t0=predicted_T0*u.day, duration=0.15*u.day) )

out_of_transit.imshow()
'define the appropriate arrays'
_meanOOTspec = out_of_transit.get_average_spectrum_as_rainbow()
meanOOTspec = _meanOOTspec.flux
meanOOTspec_relative_err = _meanOOTspec.uncertainty.value/meanOOTspec.value
oot_spec_err = meanOOTspec_relative_err.flatten()
_SED_wavelengths = (_meanOOTspec.wavelength.value).flatten()
e_per_s = meanOOTspec / exptime
e_per_s_per_angstrom = e_per_s / binwidth
_w, _s, _e = read_sensitivity_curve(grism=grism)
binned_filter_response = bintogrid(_w.value, _s.value, newx=_SED_wavelengths)['y'] * u.cm**2 / u.erg
_calibrated_mean_spec = e_per_s_per_angstrom.flatten() / binned_filter_response
calibrated_mean_spec = _calibrated_mean_spec.value/np.nanmean(_calibrated_mean_spec.value)
oot_spec_err = oot_spec_err * calibrated_mean_spec

# Convert to JAX arrays
calibrated_mean_spec_G102 = jnp.array(calibrated_mean_spec)
oot_spec_err_G102 = jnp.array(oot_spec_err)
SED_wavelengths_G102 = jnp.array(_SED_wavelengths)

visit = 'F21'
err_factor = F21_SED_err_factor

'Things that automatically get re-defined for either visit'
predicted_T0 = visits[f'{visit}']['T0 (BJD_TDB)'].value
binwidth = visits[f'{visit}']['native resolution']
exptime = visits[f'{visit}']['exp (s)']
grism = visits[f'{visit}']['Grism']
rainbow = read_rainbow(f"../../data/rainbows/{visit}_scan-combined_trimmed_pacman_spec.rainbow.npy")
speclc_rainbow = rainbow

assert(len(err_factor)==len(speclc_rainbow.wavelength))

'Adjust uncertainties and bin the rainbow'
for i in range(len(rainbow.wavelength.value)):
    speclc_rainbow.uncertainty[i,:] = rainbow.uncertainty[i,:] #* err_factor[i]

'remove the transit'
out_of_transit = ( speclc_rainbow.trim().mask_transit(period=8.463*u.day, t0=predicted_T0*u.day, duration=0.15*u.day) )

'define the appropriate arrays'
_meanOOTspec = out_of_transit.get_average_spectrum_as_rainbow()
meanOOTspec = _meanOOTspec.flux
meanOOTspec_relative_err = _meanOOTspec.uncertainty.value/meanOOTspec.value
_oot_spec_err = meanOOTspec_relative_err.flatten()
_SED_wavelengths = (_meanOOTspec.wavelength.value).flatten()
e_per_s = meanOOTspec / exptime
e_per_s_per_angstrom = e_per_s / binwidth
_w, _s, _e = read_sensitivity_curve(grism=grism)
binned_filter_response = bintogrid(_w.value, _s.value, newx=_SED_wavelengths)['y'] * u.cm**2 / u.erg
_calibrated_mean_spec = e_per_s_per_angstrom.flatten() / binned_filter_response
calibrated_mean_spec = _calibrated_mean_spec.value/np.nanmean(_calibrated_mean_spec.value)
oot_spec_err = _oot_spec_err * calibrated_mean_spec

# Convert to JAX arrays
calibrated_mean_spec_G141 = jnp.array(calibrated_mean_spec)
oot_spec_err_G141 = jnp.array(oot_spec_err)
SED_wavelengths_G141 = jnp.array(_SED_wavelengths)

plt.figure()
plt.errorbar(SED_wavelengths_G102,calibrated_mean_spec_G102,yerr=oot_spec_err_G102*100,fmt='o',label='G102',ms=1)
plt.errorbar(SED_wavelengths_G141,calibrated_mean_spec_G141,yerr=oot_spec_err_G141*100,fmt='o',label='G141',ms=1)
plt.legend()
plt.show()
plt.clf()

In [ ]:
def numpyro_model(N_temps=2, return_max_likelihood=False):
    """
    Define the probabilistic model in NUMPYRO with N_temps components.
    
    Args:
        N_temps: Number of temperature components (1, 2, 3, or 4)
        return_max_likelihood: If True, also return the maximum log likelihood value
    """
    # This function is the actual model that NumPyro will call
    def model():
        # Temperature parameters - T1 now in higher range (3700-4500)
        temp_bounds = [
            (3700, 4000),   # T1 bounds (higher temperature)
            (2000, 3700),   # T2 bounds (cooler component)
            (2000, 2900),   # T3 bounds 
        ]
        
        # Sample temperatures
        T_samples = []
        for i in range(1, N_temps + 1):
            if i == 2:
                delta_T2 = numpyro.sample('delta_T2', dist.Uniform(-900,-500))
                T = numpyro.deterministic(f'T{i}', T_samples[0] + delta_T2)
            # elif i == 3:
                # delta_T3 = numpyro.sample('delta_T3', dist.Uniform(-1700,0))
                # T = numpyro.deterministic(f'T{i}', T_samples[0] + delta_T3)
            else:
                T = numpyro.sample(f'T{i}', dist.Uniform(*temp_bounds[i-1]))
                
            T_samples.append(T)
        
        # Sample scale factors for components 2..N
        ff_samples = [1.0]  # First component always has f=1
        for i in range(2, N_temps + 1):
            ff = 10**numpyro.sample(f'ff{i}', dist.Uniform(-2, 2))
            ff_samples.append(ff)

        # Other parameters
        offset_G102 = numpyro.sample('offset_G102', dist.Uniform(0.2, 0.5))
        
        # NEW: Slope parameters for each visit (negative slope range)
        slope_G141 = numpyro.sample('slope_G141', dist.Uniform(-0.6, 0.6))
        # slope_G102 = 2*slope_G141
        slope_G102 = numpyro.sample('slope_G102', dist.Uniform(-0.6, 0.6))

        sigma_conv = 1.0 #numpyro.sample('sigma_conv', dist.Uniform(0.1, 8))
        beta_102 = numpyro.sample('beta_G102', dist.Uniform(1.0, 2.8))
        beta_141 = numpyro.sample('beta_G141', dist.Uniform(1.0, 2.8))

        # Adjust and concatenate data with slopes applied
        offset_102 = jnp.asarray(offset_G102, dtype=jnp.float64)
        
        # Apply offset and slope to G102 data (slope applied as function of index position)
        spec_G102_with_offset = jnp.array(calibrated_mean_spec_G102 + offset_102)
        
        # Apply the slopes
        spec_G102_with_slope = spec_G102_with_offset + slope_G102 * (SED_wavelengths_G102-SED_wavelengths_G102[0])
        spec_G141_with_slope = jnp.array(calibrated_mean_spec_G141) + slope_G141 * (SED_wavelengths_G141-SED_wavelengths_G141[0])
        
        # Concatenate the spectra
        _spec_flux_jax = jnp.concatenate([spec_G102_with_slope, spec_G141_with_slope])
        
        spec_flux_jax = _spec_flux_jax / jnp.mean(_spec_flux_jax)
        spec_err_jax = jnp.concatenate(
            [
                jnp.array((10**beta_102) * oot_spec_err_G102),
                jnp.array((10**beta_141) * oot_spec_err_G141)
            ]
        )
        combined_SED_wavelengths = jnp.concatenate([jnp.array(SED_wavelengths_G102), 
                                                    jnp.array(SED_wavelengths_G141)])
        
        @jit
        def decomp_model(temps, scales, sigma):
            # Get spectra for all temperatures
            spectra = [get_sphinx_spectrum_jax(T=Temp, metallicity=0.3, CtoO=0.7) for Temp in temps]
            
            # Calculate combined spectrum with scaling factors
            flux_combined = scales[0] * spectra[0][1]  # Start with first component
            for i in range(1, N_temps):
                flux_combined += scales[i] * spectra[i][1]
            
            # Normalize
            _model_flux = flux_combined / jnp.mean(flux_combined)
            
            # Convolve and bin
            convolved = convolve_spectrum_jax(
                panchromatic_wavelengths, 
                _model_flux, 
                sigma=jnp.asarray(sigma, dtype=jnp.float64)
            )
            binned_model_flux = shone.bin_spectrum(
                combined_SED_wavelengths, 
                panchromatic_wavelengths, 
                convolved
            )
            model_flux = binned_model_flux / jnp.mean(binned_model_flux)
            
            return model_flux

        # Calculate model flux
        model_flux = decomp_model(T_samples, ff_samples, sigma_conv)
        
        # Define the likelihood
        log_likelihood = dist.Normal(model_flux, spec_err_jax).log_prob(spec_flux_jax).sum()
        
        # Sample the observed data
        numpyro.sample("Obs", dist.Normal(model_flux, spec_err_jax), obs=spec_flux_jax)
        
        # Track log likelihood for maximum likelihood calculation
        if return_max_likelihood:
            numpyro.deterministic('log_likelihood', log_likelihood)
    
    # If we want to return the max likelihood callback
    if return_max_likelihood:
        def get_max_likelihood(samples):
            """Extract maximum log likelihood from samples"""
            log_lik_samples = samples.get('log_likelihood', None)
            if log_lik_samples is not None:
                return float(jnp.max(log_lik_samples))
            return None
        
        return model, get_max_likelihood
    else:
        return model

In [ ]:
rng_seed = 0

def hstack_recursive(final_states, checkpoint_states):
    for key in final_states.keys():
        if isinstance(final_states[key], dict):
            hstack_recursive(final_states[key], checkpoint_states[key])
        else:
            final_states[key] = jnp.hstack([
                final_states[key], 
                checkpoint_states[key]
            ])

def print_big_message(big_message):
    print('\n\n')
    print('=' * len(big_message))
    print(big_message)
    print('=' * len(big_message))
    print('\n\n')

# 

def post_batch_viz_save(self, **kwargs):
    """
    here we define some tasks to do after each completed checkpoint:
    """
    print(f'Corner for checkpoint {self.checkpoint}')

    samples_cumulative = self.get_samples()
    corner.corner(samples_cumulative)
    plt.suptitle(f'checkpoint {self.checkpoint}')
    plt.savefig(f'../../figs/{visit}_chkpt_{self.checkpoint}_whitelight_corner.png', dpi=200)
    plt.show()
    plt.clf()
    
    # EXTRACT MEDIAN PARAMETERS DIRECTLY FROM SAMPLES (like corner plot does)
    chkpt_median_params = {}
    for param_name, param_samples in samples_cumulative.items():
        # param_samples has shape (num_chains, num_samples)
        # Flatten across chains and compute median
        flattened_samples = param_samples.flatten()
        chkpt_median_params[param_name] = float(jnp.median(flattened_samples))
    
    print("Median parameters from samples:")
    for param, value in chkpt_median_params.items():
        print(f"  {param}: {value:.6f}")
    
    # Calculate and print maximum log likelihood
    if 'log_likelihood' in samples_cumulative:
        log_lik_samples = samples_cumulative['log_likelihood'].flatten()
        max_log_lik = float(jnp.max(log_lik_samples))
        print(f"Maximum log likelihood: {max_log_lik:.6f}")
        
        # Also compute Bayesian Information Criterion (BIC) if you want
        n_params = len([p for p in samples_cumulative.keys() if p != 'log_likelihood'])
        n_data = len(SED_wavelengths_G102) + len(SED_wavelengths_G141)  # number of data points
        bic = -2 * max_log_lik + n_params * jnp.log(n_data)
        print(f"BIC: {bic:.6f}")

    with open(f'../../data/samples/samples_cumulative_{self.start_time}_checkpoint_{self.checkpoint:04d}.pkl', 'wb') as file:
        pickle.dump(dict(samples_cumulative), file)

class MCMCWithCheckpoints(MCMC):
    running_states = None
    checkpoint = 0
    start_time = None
    
    def run_checkpoints(self, rng_key, *args, extra_fields=(), n_checkpoints=10, 
                        progress_bar_warmup=True, progress_bar_samples=True, 
                        init_params=None, on_checkpoint=None, **kwargs):
        """
        Run the MCMC samplers and collect samples.

        :param random.PRNGKey rng_key: Random number generator key to be used for the sampling.
            For multi-chains, a batch of `num_chains` keys can be supplied. If `rng_key`
            does not have batch_size, it will be split in to a batch of `num_chains` keys.
        :param args: Arguments to be provided to the :meth:`numpyro.infer.mcmc.MCMCKernel.init` method.
            These are typically the arguments needed by the `model`.
        :param extra_fields: Extra fields (aside from `"z"`, `"diverging"`) from the
            state object (e.g. :data:`numpyro.infer.hmc.HMCState` for HMC) to be collected
            during the MCMC run. Note that subfields can be accessed using dots, e.g.
            `"adapt_state.step_size"` can be used to collect step sizes at each step. Exclude sample sites from
            collection with "~`sampler.sample_field`.`sample_site`". e.g. "~z.a" will prevent site "a" from
            being collected if you're using the NUTS sampler. To collect samples of a site "a" in the
            unconstrained space, we can specify the variable here, e.g. `extra_fields=("z.a",)`.
        :type extra_fields: tuple or list of str
        :param init_params: Initial parameters to begin sampling. The type must be consistent
            with the input type to `potential_fn` provided to the kernel. If the kernel is
            instantiated by a numpyro model, the initial parameters here correspond to latent
            values in unconstrained space.
        :param kwargs: Keyword arguments to be provided to the :meth:`numpyro.infer.mcmc.MCMCKernel.init`
            method. These are typically the keyword arguments needed by the `model`.

        .. note:: jax allows python code to continue even when the compiled code has not finished yet.
            This can cause troubles when trying to profile the code for speed.
            See https://jax.readthedocs.io/en/latest/async_dispatch.html and
            https://jax.readthedocs.io/en/latest/profiling.html for pointers on profiling jax programs.
        """
        self.start_time = datetime.now().strftime("%Y-%m-%d_%H-%M")
        num_warmup_total = int(self.num_warmup)
        num_samples_total = int(self.num_samples)
        
        check_point_indices = [
            jnp.arange(num_warmup_total), 
            *jnp.array_split(jnp.arange(num_samples_total), n_checkpoints)
        ]
        n_checkpoints = len(check_point_indices)
        rng_keys = random.split(rng_key, n_checkpoints)
        pbar = tqdm(enumerate(zip(rng_keys, check_point_indices)), total=n_checkpoints)
        for checkpoint, (rng_key, bounds) in pbar:
            if checkpoint == 0:
                self.progress_bar = progress_bar_warmup
                pbar.set_description('Run warmup')
                print_big_message("Begin warmup")
                self.warmup(rng_key, *args, extra_fields=extra_fields, init_params=init_params, **kwargs)
                print_big_message(f"Begin {num_samples_total} samples with {n_checkpoints} checkpoints")

            else:
                pbar.set_description(f'Run samples {bounds.min()} to {bounds.max()}')

                self.progress_bar = progress_bar_samples
                self.num_samples = bounds.size
                self.run(rng_key, *args, extra_fields=extra_fields, init_params=init_params, **kwargs)

                # add to running states:
                if self.running_states is None:
                    self.running_states = dict(self._states)
                else:
                    hstack_recursive(self.running_states, self._states)
                
                # ensure that calls to `self.get_samples` will build a new samples array
                # out of the running states:
                self._states_flat = None
                self._states = self.running_states

                if on_checkpoint is not None:
                    on_checkpoint(self, **kwargs)
                self.checkpoint += 1
        
        pbar.close()

        # reset to total number for arviz IO
        self.num_samples = num_samples_total

In [ ]:
n_temps = 2
n_warmup = 4_00
n_samples = 1_000
n_chains = 8
n_check = 2

# Get the model function and callback
model_func, get_max_likelihood = numpyro_model(N_temps=n_temps, return_max_likelihood=True)

model_designation = f'jointvisit_{n_temps}T_{n_samples}_{n_chains}chains_SED_sphinx'

rng_key = PRNGKey(42)
# rng_keys = random.split(rng_key, n_chains)

sampler = NUTS(
    model_func,  # Use the returned function
    dense_mass=True
)

mcmc = MCMCWithCheckpoints(
    sampler, 
    num_warmup=n_warmup, 
    num_samples=n_samples,
    num_chains=n_chains
)

mcmc.run_checkpoints(rng_key, n_checkpoints=n_check, on_checkpoint=post_batch_viz_save)

In [ ]:
mcmc.print_summary()

# Extract maximum likelihood after sampling
if get_max_likelihood is not None:
    samples = mcmc.get_samples()
    max_log_lik = get_max_likelihood(samples)
    print(f"Maximum log likelihood: {max_log_lik:.6f}")

result = arviz.from_numpyro(mcmc)

# Post-Inference Analysis

In [ ]:
'Save the result'
result.to_netcdf(f'../../data/samples/{model_designation}')

'Print the summary'
arviz.summary(result)

In [ ]:
'Examine the Leave-One-Out (LOO) summary'
loo = arviz.loo(result, pointwise=True)
loo

In [ ]:
'Extract median result'
median_params = arviz.summary(result,stat_focus='median')['median']

In [ ]:
'Make a corner plot'
corner.corner(
    result, var_names=['~offset_G102',
                       r'~$\beta_{\rm G102}$',
                       r'~$\beta_{\rm G141}$',
                       r'~$\sigma_{\rm conv}$'],
    # quiet=True, 
);
plt.savefig(f'../../figs/{model_designation}_corner.png',dpi=200)

In [ ]:
# Get the samples from your MCMC run
samples = mcmc.get_samples()  # This gives you all samples

max_likelihood_idx = jnp.argmax(samples['log_likelihood'])

# Extract the maximum likelihood parameters
max_likelihood_params = {}
for key in samples.keys():
    max_likelihood_params[key] = samples[key][max_likelihood_idx]

# Now use max_likelihood_params instead of median_params in your plotting code
spec_G102 = jnp.array(calibrated_mean_spec_G102 + jnp.asarray(max_likelihood_params['offset_G102'], dtype=jnp.float64))
spec_G141 = jnp.array(calibrated_mean_spec_G141)

_spec_flux_jax = jnp.concatenate([
    spec_G102 + max_likelihood_params['slope_G102'] * (SED_wavelengths_G102 - SED_wavelengths_G102[0]),
    spec_G141 + max_likelihood_params['slope_G141'] * (SED_wavelengths_G141 - SED_wavelengths_G141[0]) 
])

max_likelihood_spec_flux = _spec_flux_jax / jnp.mean(_spec_flux_jax)
max_likelihood_spec_err = jnp.concatenate([
    jnp.array(oot_spec_err_G102 * jnp.asarray(10**max_likelihood_params[r'beta_G102'], dtype=jnp.float64)),
    jnp.array(oot_spec_err_G141 * jnp.asarray(10**max_likelihood_params[r'beta_G141'], dtype=jnp.float64))
])
combined_SED_wavelengths = jnp.concatenate([jnp.array(SED_wavelengths_G102), jnp.array(SED_wavelengths_G141)])

filter_sigma = 1.0  #max_likelihood_params['sigma_conv']

temps = []
for i in range(1, n_temps + 1):
    temp_key = f'T{i}'
    temps.append(jnp.asarray(max_likelihood_params[temp_key], dtype=jnp.float64))

# Get spectra for all temperature components
spectra = [get_sphinx_spectrum_jax(T=Temp, metallicity=0.3, CtoO=0.7) for Temp in temps]

# Get scale factors
scales = [1.0]
for i in range(2, n_temps + 1):
    ff_key = f'ff{i}'
    scale = jnp.asarray(10**max_likelihood_params[ff_key], dtype=jnp.float64)
    scales.append(scale)

# Calculate combined spectrum
_max_likelihood_flux = scales[0] * spectra[0][1]
for i in range(1, n_temps):
    _max_likelihood_flux += scales[i] * spectra[i][1]

_model_flux = _max_likelihood_flux / jnp.mean(_max_likelihood_flux)
convolved = convolve_spectrum_jax(panchromatic_wavelengths, _model_flux, sigma=filter_sigma)
binned_model_flux = shone.bin_spectrum(combined_SED_wavelengths, panchromatic_wavelengths, convolved)
max_likelihood_model_flux = (binned_model_flux / jnp.mean(binned_model_flux))

# Plot with maximum likelihood parameters
fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
fig.suptitle(r'0.8-1.64 $\mu$m SED with ' + f'{n_temps}T Model (Maximum Likelihood)', fontsize=20)
axs[1].set_title('Residuals', fontsize=14)
axs[1].set_xlim(0.805, 1.645)
axs[1].set_ylim(-4, 4)
axs[1].set_xlabel(r'Wavelength ($\mu$m)', fontsize=14)
axs[0].set_ylabel('Relative Flux', fontsize=14)
axs[1].set_ylabel(r'$\sigma$', fontsize=14)

axs[0].errorbar(combined_SED_wavelengths,
                max_likelihood_spec_flux,
                yerr=max_likelihood_spec_err,
                fmt='o', color='b', ms=1, alpha=0.7)
axs[0].plot(combined_SED_wavelengths,
            max_likelihood_model_flux, color='k', zorder=100)
axs[1].scatter(combined_SED_wavelengths,
               (max_likelihood_model_flux - max_likelihood_spec_flux) / max_likelihood_spec_err,
               color='gray')
axs[1].axhline(0, color='k', zorder=-100)

plt.savefig(f'../../figs/{model_designation}_max_likelihood_result.png', dpi=200)

In [ ]:
# 'Plot the data, model, and residuals'

# spec_G102 = jnp.array(calibrated_mean_spec_G102 + jnp.asarray(median_params['offset_G102'], dtype=jnp.float64))
# spec_G141 = jnp.array(calibrated_mean_spec_G141)

# _spec_flux_jax = jnp.concatenate([
#     spec_G102 + median_params['slope_G102'] * (SED_wavelengths_G102-SED_wavelengths_G102[0]),
#     spec_G141 + median_params['slope_G141'] * (SED_wavelengths_G141-SED_wavelengths_G141[0]) 
# ])

# median_spec_flux = _spec_flux_jax / jnp.mean(_spec_flux_jax)
# median_spec_err = jnp.concatenate([jnp.array(oot_spec_err_G102 * jnp.asarray(10**median_params[r'beta_G102'], dtype=jnp.float64)),
#                                     jnp.array(oot_spec_err_G141 * jnp.asarray(10**median_params[r'beta_G141'], dtype=jnp.float64))])
# combined_SED_wavelengths = jnp.concatenate([jnp.array(SED_wavelengths_G102), jnp.array(SED_wavelengths_G141)])

# filter_sigma = 1.0 #jnp.asarray(median_params[r'sigma_conv'], dtype=jnp.float64)

# temps = []
# for i in range(1, n_temps + 1):
#     temp_key = f'T{i}'  # Note: using T1, T2, T3, T4 format
#     temps.append(jnp.asarray(median_params[temp_key], dtype=jnp.float64))

# # Get spectra for all temperature components
# spectra = [get_sphinx_spectrum_jax(T=Temp, metallicity=0.3, CtoO=0.7) for Temp in temps]

# # Get scale factors (first component = 1, others from ff parameters)
# scales = [1.0]  # First component always has scale 1
# for i in range(2, n_temps + 1):
#     ff_key = f'ff{i}'  # Note: using ff2, ff3, ff4 format
#     scale = jnp.asarray(10**median_params[ff_key], dtype=jnp.float64)
#     scales.append(scale)

# # Calculate combined spectrum with scaling factors
# _median_flux = scales[0] * spectra[0][1]  # Start with first component
# for i in range(1, n_temps):
#     _median_flux += scales[i] * spectra[i][1]

# _model_flux = _median_flux / jnp.mean(_median_flux)
# convolved = convolve_spectrum_jax(panchromatic_wavelengths, _model_flux, sigma=filter_sigma)
# binned_model_flux = shone.bin_spectrum(combined_SED_wavelengths, panchromatic_wavelengths, convolved)
# median_model_flux = (binned_model_flux / jnp.mean(binned_model_flux))

# fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
# fig.suptitle(r'0.8-1.64 $\mu$m SED with ' + f'{n_temps}T Model', fontsize=20)
# axs[1].set_title('Residuals', fontsize=14)
# axs[1].set_xlim(0.805, 1.645)
# axs[1].set_ylim(-4, 4)
# axs[1].set_xlabel(r'Wavelength ($\mu$m)', fontsize=14)
# axs[0].set_ylabel('Relative Flux', fontsize=14)
# axs[1].set_ylabel(r'$\sigma$', fontsize=14)

# axs[0].errorbar(combined_SED_wavelengths,
#                 median_spec_flux,
#                 yerr=median_spec_err,
#                 fmt='o', color='b', ms=1, alpha=0.7)
# axs[0].plot(combined_SED_wavelengths,
#             median_model_flux, color='k', zorder=100)
# axs[1].scatter(combined_SED_wavelengths,
#                (median_model_flux - median_spec_flux) / median_spec_err,
#                color='gray')
# axs[1].axhline(0,color='k',zorder=-100)

# plt.savefig(f'../../figs/{model_designation}_result.png', dpi=200)

In [ ]:
n_temps = 3
n_warmup = 4_00
n_samples = 1_000
n_chains = 8
n_check = 2

# Get the model function and callback
model_func, get_max_likelihood = numpyro_model(N_temps=n_temps, return_max_likelihood=True)

model_designation = f'jointvisit_{n_temps}T_{n_samples}_{n_chains}chains_SED_sphinx'

rng_key = PRNGKey(42)
# rng_keys = random.split(rng_key, n_chains)

sampler = NUTS(
    model_func,  # Use the returned function
    dense_mass=True
)

mcmc = MCMCWithCheckpoints(
    sampler, 
    num_warmup=n_warmup, 
    num_samples=n_samples,
    num_chains=n_chains
)

mcmc.run_checkpoints(rng_key, n_checkpoints=n_check, on_checkpoint=post_batch_viz_save)

mcmc.print_summary()

# Extract maximum likelihood after sampling
if get_max_likelihood is not None:
    samples = mcmc.get_samples()
    max_log_lik = get_max_likelihood(samples)
    print(f"Maximum log likelihood: {max_log_lik:.6f}")

result = arviz.from_numpyro(mcmc)

In [ ]:
'Save the result'
result.to_netcdf(f'../../data/samples/{model_designation}')

'Print the summary'
arviz.summary(result)

'Examine the Leave-One-Out (LOO) summary'
loo = arviz.loo(result, pointwise=True)
loo

# 'Plot the data, model, and residuals'

# spec_G102 = jnp.array(calibrated_mean_spec_G102 + jnp.asarray(median_params['offset_G102'], dtype=jnp.float64))
# spec_G141 = jnp.array(calibrated_mean_spec_G141)

# _spec_flux_jax = jnp.concatenate([
#     spec_G102 + 2* median_params['slope_G141'] * (SED_wavelengths_G102-SED_wavelengths_G102[0]),
#     spec_G141 + median_params['slope_G141'] * (SED_wavelengths_G141-SED_wavelengths_G141[0]) 
# ])

# # _spec_flux_jax = jnp.concatenate([
# #     spec_G102 -0.1 * (SED_wavelengths_G102-SED_wavelengths_G102[0]),
# #     spec_G141 -0.06 * (SED_wavelengths_G141-SED_wavelengths_G141[0]) 
# # ])

# median_spec_flux = _spec_flux_jax / jnp.mean(_spec_flux_jax)
# median_spec_err = jnp.concatenate([jnp.array(oot_spec_err_G102 * jnp.asarray(10**median_params[r'beta_G102'], dtype=jnp.float64)),
#                                     jnp.array(oot_spec_err_G141 * jnp.asarray(10**median_params[r'beta_G141'], dtype=jnp.float64))])
# combined_SED_wavelengths = jnp.concatenate([jnp.array(SED_wavelengths_G102), jnp.array(SED_wavelengths_G141)])

# filter_sigma = jnp.asarray(2, dtype=jnp.float64)

# # Dynamically get all temperature components
# temps = []
# for i in range(1, n_temps + 1):
#     temp_key = f'T{i}'  # Note: using T1, T2, T3, T4 format
#     temps.append(jnp.asarray(median_params[temp_key], dtype=jnp.float64))

# # Get spectra for all temperature components
# spectra = [get_sphinx_spectrum_jax(T=Temp, metalllicity=0.0, CtoO=0.9) for Temp in temps]


# # Get scale factors (first component = 1, others from ff parameters)
# scales = [1.0]  # First component always has scale 1
# for i in range(2, n_temps + 1):
#     ff_key = f'ff{i}'  # Note: using ff2, ff3, ff4 format
#     scale = jnp.asarray(10**median_params[ff_key], dtype=jnp.float64)
#     scales.append(scale)

# # Calculate combined spectrum with scaling factors
# _median_flux = scales[0] * spectra[0][1]  # Start with first component
# for i in range(1, n_temps):
#     _median_flux += scales[i] * spectra[i][1]

# _model_flux = _median_flux / jnp.mean(_median_flux)
# convolved = convolve_spectrum_jax(panchromatic_wavelengths, _model_flux, sigma=filter_sigma)
# binned_model_flux = shone.bin_spectrum(combined_SED_wavelengths, panchromatic_wavelengths, convolved)
# median_model_flux = (binned_model_flux / jnp.mean(binned_model_flux))

# fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
# fig.suptitle(r'0.8-1.64 $\mu$m SED with ' + f'{n_temps}T Model', fontsize=20)
# axs[1].set_title('Residuals', fontsize=14)
# axs[1].set_xlim(0.805, 1.645)
# axs[1].set_ylim(-4, 4)
# axs[1].set_xlabel(r'Wavelength ($\mu$m)', fontsize=14)
# axs[0].set_ylabel('Relative Flux', fontsize=14)
# axs[1].set_ylabel(r'$\sigma$', fontsize=14)

# axs[0].errorbar(combined_SED_wavelengths,
#                 median_spec_flux,
#                 yerr=median_spec_err,
#                 fmt='o', color='b', ms=1, alpha=0.7)
# axs[0].plot(combined_SED_wavelengths,
#             median_model_flux, color='k', zorder=100)
# axs[1].scatter(combined_SED_wavelengths,
#                (median_model_flux - median_spec_flux) / median_spec_err,
#                color='gray')
# axs[1].axhline(0,color='k',zorder=-100)

# plt.savefig(f'../../figs/{model_designation}_result.png', dpi=200)

In [ ]:
# Get the samples from your MCMC run
samples = mcmc.get_samples()  # This gives you all samples
log_likelihood = mcmc.get_log_likelihood()  # This gives you log likelihood for each sample

# Find the index of the maximum likelihood sample
# If you have log likelihood directly:
max_likelihood_idx = jnp.argmax(jnp.sum(log_likelihood, axis=-1))  # Sum over data points if needed

# Or if you have log probability from the sampler:
# max_likelihood_idx = jnp.argmax(samples['potential_energy'])  # Lower potential energy = higher probability

# Extract the maximum likelihood parameters
max_likelihood_params = {}
for key in samples.keys():
    max_likelihood_params[key] = samples[key][max_likelihood_idx]

# Now use max_likelihood_params instead of median_params in your plotting code
spec_G102 = jnp.array(calibrated_mean_spec_G102 + jnp.asarray(max_likelihood_params['offset_G102'], dtype=jnp.float64))
spec_G141 = jnp.array(calibrated_mean_spec_G141)

_spec_flux_jax = jnp.concatenate([
    spec_G102 + max_likelihood_params['slope_G102'] * (SED_wavelengths_G102 - SED_wavelengths_G102[0]),
    spec_G141 + max_likelihood_params['slope_G141'] * (SED_wavelengths_G141 - SED_wavelengths_G141[0]) 
])

max_likelihood_spec_flux = _spec_flux_jax / jnp.mean(_spec_flux_jax)
max_likelihood_spec_err = jnp.concatenate([
    jnp.array(oot_spec_err_G102 * jnp.asarray(10**max_likelihood_params[r'beta_G102'], dtype=jnp.float64)),
    jnp.array(oot_spec_err_G141 * jnp.asarray(10**max_likelihood_params[r'beta_G141'], dtype=jnp.float64))
])
combined_SED_wavelengths = jnp.concatenate([jnp.array(SED_wavelengths_G102), jnp.array(SED_wavelengths_G141)])

filter_sigma = 1.0  # or max_likelihood_params['sigma_conv'] if you have it

temps = []
for i in range(1, n_temps + 1):
    temp_key = f'T{i}'
    temps.append(jnp.asarray(max_likelihood_params[temp_key], dtype=jnp.float64))

# Get spectra for all temperature components
spectra = [get_sphinx_spectrum_jax(T=Temp, metallicity=0.3, CtoO=0.7) for Temp in temps]

# Get scale factors
scales = [1.0]
for i in range(2, n_temps + 1):
    ff_key = f'ff{i}'
    scale = jnp.asarray(10**max_likelihood_params[ff_key], dtype=jnp.float64)
    scales.append(scale)

# Calculate combined spectrum
_max_likelihood_flux = scales[0] * spectra[0][1]
for i in range(1, n_temps):
    _max_likelihood_flux += scales[i] * spectra[i][1]

_model_flux = _max_likelihood_flux / jnp.mean(_max_likelihood_flux)
convolved = convolve_spectrum_jax(panchromatic_wavelengths, _model_flux, sigma=filter_sigma)
binned_model_flux = shone.bin_spectrum(combined_SED_wavelengths, panchromatic_wavelengths, convolved)
max_likelihood_model_flux = (binned_model_flux / jnp.mean(binned_model_flux))

# Plot with maximum likelihood parameters
fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
fig.suptitle(r'0.8-1.64 $\mu$m SED with ' + f'{n_temps}T Model (Maximum Likelihood)', fontsize=20)
axs[1].set_title('Residuals', fontsize=14)
axs[1].set_xlim(0.805, 1.645)
axs[1].set_ylim(-4, 4)
axs[1].set_xlabel(r'Wavelength ($\mu$m)', fontsize=14)
axs[0].set_ylabel('Relative Flux', fontsize=14)
axs[1].set_ylabel(r'$\sigma$', fontsize=14)

axs[0].errorbar(combined_SED_wavelengths,
                max_likelihood_spec_flux,
                yerr=max_likelihood_spec_err,
                fmt='o', color='b', ms=1, alpha=0.7)
axs[0].plot(combined_SED_wavelengths,
            max_likelihood_model_flux, color='k', zorder=100)
axs[1].scatter(combined_SED_wavelengths,
               (max_likelihood_model_flux - max_likelihood_spec_flux) / max_likelihood_spec_err,
               color='gray')
axs[1].axhline(0, color='k', zorder=-100)

plt.savefig(f'../../figs/{model_designation}_max_likelihood_result.png', dpi=200)

# Compare multiple models of the same dataset

In [ ]:
# compare_dict = {}

# compare_dict['model_1'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_3T_1000_11chains_SED_separatebetas')
# compare_dict['model_2'] = arviz.InferenceData.from_netcdf('../../data/samples/jointvisit_3T_1000_14chains_SED_photonerrs')
# # compare_dict['model_3'] = arviz.InferenceData.from_netcdf('../../data/samples/')

# arviz.compare(compare_dict)

From https://num.pyro.ai/en/latest/tutorials/bad_posterior_geometry.html

In general it is difficult to assess whether the samples returned from HMC or NUTS represent accurate (approximate) samples from the posterior. Two general rules of thumb, however, are to look at the effective sample size (ESS) and r_hat diagnostics returned by mcmc.print_summary(). If we see values of r_hat in the range (1.0, 1.05) and effective sample sizes that are comparable to the total number of samples num_samples (assuming thinning=1) then we have good reason to believe that HMC is doing a good job. If, however, we see low effective sample sizes or large r_hats for some of the variables (e.g. r_hat = 1.15) then HMC is likely struggling with the posterior geometry. In the following we will use r_hat as our primary diagnostic metric.